**Import Libraries**

In [1]:
import os
import torch
import pandas as pd
import librosa
from transformers import pipeline
from sentence_transformers import SentenceTransformer
from tqdm.notebook import tqdm
import warnings

warnings.filterwarnings("ignore")

**Load Both Models (ASR + Embedder)**

In [2]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

print("Loading Whisper ASR model...")   # Speech-to-Text Model (Whisper)
asr_pipeline = pipeline(
    "automatic-speech-recognition", 
    model="openai/whisper-tiny.en", 
    device=device
)

print("Loading BAAI/bge-base-en-v1.5 model...") # Text Embedding Model (BGE)
text_model = SentenceTransformer('BAAI/bge-base-en-v1.5', device=device)

print("Both models loaded successfully!")

Using device: cuda:0
Loading Whisper ASR model...


Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

Loading BAAI/bge-base-en-v1.5 model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Both models loaded successfully!


**Process the Sequential Chunks**

In [4]:
df_chunks = pd.read_csv("../../data/processed/sequential_metadata.csv")

features_list = []
valid_call_ids = []
valid_chunk_indices = []
valid_labels = []
transcribed_texts = []

print(f"Transcribing and embedding {len(df_chunks)} audio chunks...")

for index, row in tqdm(df_chunks.iterrows(), total=len(df_chunks)):
    file_path = row['chunk_path']
    
    if os.path.exists(file_path):
        try:
            y, sr = librosa.load(file_path, sr=16000)
            transcription = asr_pipeline({"sampling_rate": sr, "raw": y})['text'].strip()
            embedding = text_model.encode(transcription)
            
            features_list.append(embedding)
            valid_call_ids.append(row['original_call_id'])
            valid_chunk_indices.append(row['chunk_index'])
            valid_labels.append(row['label'])
            transcribed_texts.append(transcription)
            
        except Exception as e:
            print(f"Error processing {file_path}: {e}")

print(f"Successfully processed {len(features_list)} text embeddings.")

Transcribing and embedding 3958 audio chunks...


  0%|          | 0/3958 [00:00<?, ?it/s]

Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'tr

Successfully processed 3958 text embeddings.


**Save the Features to Parquet**

In [6]:
embedding_dim = len(features_list[0])
col_names = [f'text_dim_{i}' for i in range(1, embedding_dim + 1)]

text_features_df = pd.DataFrame(features_list, columns=col_names)
text_features_df.insert(0, 'original_call_id', valid_call_ids)
text_features_df.insert(1, 'chunk_index', valid_chunk_indices)
text_features_df.insert(2, 'label', valid_labels)
text_features_df['chunk_text'] = transcribed_texts 

final_path = "../../data/processed/text_features.parquet"
text_features_df.to_parquet(final_path, index=False, engine='pyarrow')

print(f"Saved sequential text features to: {final_path}")
text_features_df.head()

Saved sequential text features to: ../../data/processed/text_features.parquet


,original_call_id,chunk_index,label,text_dim_1,text_dim_2,text_dim_3,text_dim_4,text_dim_5,text_dim_6,text_dim_7,...,text_dim_760,text_dim_761,text_dim_762,text_dim_763,text_dim_764,text_dim_765,text_dim_766,text_dim_767,text_dim_768,chunk_text
0,sample_0,0,0,-0.034074,0.000385,0.007431,-0.049374,-0.014718,0.011791,-0.020818,...,-0.021365,0.022887,-0.050455,-0.003284,-0.041325,-0.011346,0.031532,0.040242,0.002005,Greetings. This is. Name. I finally got my han...
1,sample_0,1,0,-0.011793,0.042084,0.045106,-0.012623,0.029359,0.036353,0.007132,...,-0.002989,0.031355,-0.045678,0.054876,-0.001477,0.016482,0.017949,0.014928,-0.044811,product. You mentioned last month. It is as go...
2,sample_0,2,0,0.019739,0.022258,0.035350,0.013133,0.005481,-0.021418,0.051408,...,-0.012205,0.006638,-0.068714,-0.032664,0.024762,0.017319,-0.028105,-0.004401,-0.045563,Let me know when you want to try it out.
3,sample_1,0,0,-0.049042,0.023259,0.037097,-0.002572,0.015321,0.013329,0.004865,...,-0.048373,0.018726,-0.061888,0.014444,-0.007830,-0.026024,0.022285,0.047761,0.029023,Greetings. This is. Name. I am hosting a small...
4,sample_1,1,0,-0.020365,0.014099,0.084816,-0.021772,0.010355,-0.031146,0.060457,...,-0.017148,0.012448,-0.070382,-0.056377,-0.005714,-0.018005,-0.022768,0.030692,-0.008290,this date at time. It will be a relaxed evenin...
